In [1]:
import os

from vfs.backends.postgres import PostgresFileSystem
from vfs.models import VFSEntry
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm

In [2]:
db_name = 'pg_trgm_eval'
db_url = f'postgresql+asyncpg://localhost/{db_name}'

SKIP_DIRS = {
    '.git', '.hg', '.svn', '__pycache__', '.venv', 'venv', 'env',
    '.tox', '.pytest_cache', '.mypy_cache', '.ruff_cache', '.ipynb_checkpoints',
    'node_modules', '.next', '.turbo', '.idea', '.vscode', 'build', 'dist', 'target',
    'out', '.cache', '.gradle', '.terraform',
}

BINARY_EXTS = {
    '.pyc', '.pyo', '.so', '.dll', '.dylib', '.exe', '.bin',
    '.png', '.jpg', '.jpeg', '.gif', '.bmp', '.ico', '.webp', '.tiff',
    '.pdf', '.zip', '.tar', '.gz', '.bz2', '.xz', '.7z', '.rar',
    '.mp3', '.mp4', '.wav', '.avi', '.mov', '.mkv', '.webm',
    '.ttf', '.otf', '.woff', '.woff2',
    '.db', '.sqlite', '.sqlite3',
    '.parquet', '.arrow', '.feather',
}

In [19]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1024, chunk_overlap=128)

In [20]:
paths = []
for dirpath, dirnames, filenames in os.walk('/Users/claygendron/Git/Repos/grover/'):
    dirnames[:] = [d for d in dirnames if d not in SKIP_DIRS]
    for fn in filenames:
        if os.path.splitext(fn)[1].lower() not in BINARY_EXTS:
            paths.append(os.path.join(dirpath, fn))



In [21]:
for path in tqdm(paths, desc='reading', unit='file'):
    try:
        with open(path, encoding='utf-8') as f:
            data = f.read()
    except (OSError, UnicodeDecodeError):
        data = ''

    data = '' if '\x00' in data else data

    if data != '':
        file_entry = VFSEntry(path=path, content=data)
        chunks = text_splitter.create_documents([data])
        chunk_entry =

reading: 100%|██████████| 376/376 [00:00<00:00, 403.21file/s] 


In [29]:
list1 = [0, 1, 2]
print(list1)
print(list1.pop(0))
print(list1)

[0, 1, 2]
0
[1, 2]


In [24]:
print(chunks[0].page_content)

"""Execution engine for the CLI query AST."""

from __future__ import annotations

import asyncio
from collections import deque
from typing import TYPE_CHECKING

from vfs.columns import CANDIDATE_FIELD_TO_MODEL_COLUMNS, required_model_columns
from vfs.paths import normalize_path, parse_kind
from vfs.query.ast import (
    CaseMode,
    CopyCommand,
    DeleteCommand,
    EditCommand,
    ExceptStage,
    GlobCommand,
    GraphTraversalCommand,
    GrepCommand,
    GrepOutputMode,
    IntersectStage,
    KindsCommand,
    LexicalSearchCommand,
    LsCommand,
    MeetingGraphCommand,
    MkdirCommand,
    MkedgeCommand,
    MoveCommand,
    PipelineNode,
    QueryNode,
    QueryPlan,
    RankCommand,
    ReadCommand,
    SemanticSearchCommand,
    SortCommand,
    StageNode,
    StatCommand,
    TopCommand,
    TreeCommand,
    UnionNode,
    VectorSearchCommand,
    Visibility,
    WriteCommand,
)
from vfs.results import CANDIDATE_FIELDS, PROJECTION_SENTINELS, Candidate, TwoPathOperatio